<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/04_generative_ai/llms/experiment_llama_finetuning_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Import required libraries
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch version: 2.6.0+cu124
CUDA available: True


In [ ]:
# Load the Alpaca dataset from Hugging Face
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
# Shuffle and select a small subset
train_data = dataset.shuffle(seed=42).select(range(50))

# Format each example as a single prompt string for causal LM:
# We concatenate the instruction (and input if present) with the output.
def format_prompt(example):
    if example["input"]:
        return f"Instruction: {example['instruction']} Input: {example['input']}  Response: {example['output']}"
    else:
        return f"Instruction: {example['instruction']}  Response: {example['output']}"

train_data = train_data.map(lambda x: {"text": format_prompt(x)}, remove_columns=["instruction","input","output"])
print(f"Example formatted prompt:\n{train_data[0]['text']}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Example formatted prompt:
Instruction: Rearrange the following sentence to make the sentence more interesting. Input: She left the party early  Response: Early, she left the party.


In [ ]:
model_name = "NousResearch/Llama-2-7b-chat-hf"  # LLaMA-2 7B model (HF format)

print("Loading tokenizer and model in 8-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,       # load weights in 8-bit precision
    device_map="auto",       # automatically map layers to GPU
    torch_dtype=torch.float16  # use float16 for computations
)
print("Model loaded.")

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,                     # rank of LoRA matrices
    lora_alpha=16,           # scaling factor
    target_modules=["q_proj", "k_proj", "v_proj"],  # apply to these layers
    lora_dropout=0.05,       # dropout for LoRA layers
    bias="none",             # no bias (we only adapt weights)
    task_type="CAUSAL_LM"
)
# Add LoRA adapters to the model
model = get_peft_model(model, lora_config)
print("LoRA adapter added to the model.")


Loading tokenizer and model in 8-bit...


tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model loaded.
LoRA adapter added to the model.


In [ ]:
# Tokenize the dataset
print(train_data[0])  # Print the first example


# # Tokenize function with explicit padding and truncation
# def tokenize_function(examples):
#     # If your 'text' field contains lists of strings, you might need to join them
#     # If text is a list of lists or has other nested structure, you need to flatten it
#     if isinstance(examples["text"][0], list):
#         examples["text"] = [" ".join(text) if isinstance(text, list) else text for text in examples["text"]]

#     return tokenizer(
#         examples["text"],
#         padding="max_length",
#         truncation=True,
#         max_length=512  # Set appropriate max length
#     )

# Make sure your tokenization function uses a reasonable max_length
def tokenize_function(examples):
    # Handle potential nested structures
    if isinstance(examples["text"][0], list):
        examples["text"] = [" ".join(text) if isinstance(text, list) else text for text in examples["text"]]

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256  # Reduced from larger values like 512
    )
# Apply tokenization
tokenized_train_data = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
# tokenized_train_data = tokenized_train_data.remove_columns(["attention_mask"])
# tokenized_dataset = tokenized_train_data.train_test_split(test_size=0.1)


{'text': 'Instruction: Rearrange the following sentence to make the sentence more interesting. Input: She left the party early  Response: Early, she left the party.'}


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
tokenized_train_data
tiny_train_data = tokenized_train_data.select(range(min(10, len(tokenized_train_data))))
tiny_train_data

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10
})

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Clear GPU cache first
import torch
torch.cuda.empty_cache()

# Load the model in 4-bit instead of 8-bit
from transformers import BitsAndBytesConfig

# Define quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Load model with proper quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

# Make sure model parameters are trainable
model.gradient_checkpointing_enable()  # Enable gradient checkpointing
model.enable_input_require_grads()  # This is crucial for PEFT models

# Configure LoRA correctly for quantized models
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

# Apply PEFT
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()  # Verify some parameters are trainable

# Training arguments - disable gradient checkpointing to avoid conflicts
training_args = TrainingArguments(
    output_dir="llama2-lora-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    fp16=True,
    push_to_hub=False,
    report_to="none",
    save_strategy="no",
    gradient_checkpointing=False,  # Disable this as we've enabled it on the model directly
    optim="paged_adamw_8bit"  # Use paged optimizer for memory efficiency
)

# Create a tiny test dataset
tiny_train_data = tokenized_train_data.select(range(min(5, len(tokenized_train_data))))

# Initialize trainer
trainer = Trainer(
    model=model,
    train_dataset=tiny_train_data,
    args=training_args,
    data_collator=data_collator
)

# Try training with minimal epochs
print("Starting minimal training test...")
trainer.train()
print("Test complete.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622
Starting minimal training test...


Step,Training Loss
1,3.759000
2,2.267100
3,1.415100
4,3.680500
5,1.476000


Test complete.
